# Stage 6: selection of final $k_{2,\mathrm{high}}$ values

This notebook collects the final kinetic parameters from all `*_k2_next_step_analysis.xlsx`
workbooks below the main kinetics directory containing `P5`, `P6`, ..., `P9`. It also imports the per-PX
water/hydroxide `k0` summaries and appends water as `Nu1` entries.

The main output is a wide Excel table with one row per `PX / dataset / nucleophile`.
It contains Strategy A pH-rate fit values, Strategy A high-pH plateau mean/median
summaries, Strategy B global-fit values, and synthetic `Nu1` water rows sourced from
the `k0` pH-rate analysis. Failed fits are kept as rows for auditability, but their
fitted values are written as blank/`NaN`.


## Configuration

By default, the notebook scans the directory in which the notebook is run. This directory should contain `P5`, `P6`, ..., `P9`.

By default, the summary Excel file is written directly into this root directory. You can override the root or output directory before execution with:

```bash
K2_SUMMARY_ROOT=/path/to/kinetics
K2_SUMMARY_OUTPUT_DIR=/path/to/output
```

In [8]:
from pathlib import Path
import os
import warnings

import numpy as np
import pandas as pd

DEFAULT_ROOT = Path.cwd().resolve()
ROOT = Path(os.environ.get("K2_SUMMARY_ROOT", DEFAULT_ROOT)).expanduser()
OUTPUT_DIR = Path(os.environ.get("K2_SUMMARY_OUTPUT_DIR", ROOT)).expanduser()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESULT_PATTERN = "*/S*/k2_next_step_analysis/*_k2_next_step_analysis.xlsx"
K0_RESULT_PATTERN = "P*/k0_ph_rate_P*.xlsx"

print(f"ROOT = {ROOT}")
print(f"OUTPUT_DIR = {OUTPUT_DIR}")


ROOT = /Users/robertmayer/Desktop/mayerlab_local/phosphorylation_publ/section5_6_kinetics
OUTPUT_DIR = /Users/robertmayer/Desktop/mayerlab_local/phosphorylation_publ/section5_6_kinetics


## Helpers

The individual result workbooks already carry `nu_id` and `nu_name` as propagated
from the corresponding `*_fulldata.xlsx` sheets. This summary notebook preserves those
names as-is and only adds directory-level metadata such as `PX_directory` and `dataset`.

In [9]:
KEY_COLS = ["PX_directory", "dataset", "PX", "nu_id", "nu_name"]


def electrophile_key(value):
    text = str(value).strip()
    return text.split("_", 1)[0]


def discover_result_workbooks(root=ROOT):
    paths = sorted(root.glob(RESULT_PATTERN))
    return [p for p in paths if not p.name.startswith("~$")]


def discover_k0_summary_workbooks(root=ROOT):
    return [p for p in sorted(root.glob(K0_RESULT_PATTERN)) if not p.name.startswith("~$")]


def parse_result_path(path):
    path = Path(path)
    dataset_dir = path.parents[1]
    px_dir = path.parents[2]
    return {
        "PX_directory": px_dir.name,
        "dataset": dataset_dir.name,
        "source_workbook": str(path),
    }


def read_sheet(path, sheet_name):
    try:
        return pd.read_excel(path, sheet_name=sheet_name)
    except ValueError:
        warnings.warn(f"{path.name}: missing sheet {sheet_name}")
        return pd.DataFrame()


def truthy(series):
    if series is None:
        return pd.Series(dtype=bool)
    return series.fillna(False).astype(bool)


def nu_sort_key(value):
    match = pd.Series([value]).astype(str).str.extract(r"Nu\s*(\d+)", expand=False).iloc[0]
    if pd.isna(match):
        return np.inf
    return int(match)


def with_common_metadata(df, path):
    if df.empty:
        return df
    meta = parse_result_path(path)
    out = df.copy()
    out.insert(0, "PX_directory", meta["PX_directory"])
    out.insert(1, "dataset", meta["dataset"])
    out["source_workbook"] = meta["source_workbook"]
    if "electrophile_type" in out.columns:
        out = out.rename(columns={"electrophile_type": "PX"})
    return out


def infer_px_directory(px):
    key = electrophile_key(px)
    matches = sorted(p.name for p in ROOT.glob(f"{key}*") if p.is_dir())
    return matches[0] if matches else key


def read_k0_summary_workbooks():
    rows = []
    for path in discover_k0_summary_workbooks():
        df = read_sheet(path, "summary")
        if df.empty:
            continue
        df = df.copy()
        df["k0_source_workbook"] = str(path)
        if "PX_dir" not in df.columns:
            if path.parent.name.startswith("P") and path.parent.parent == ROOT:
                df["PX_dir"] = path.parent.name
            else:
                df["PX_dir"] = df["electrophile_type"].map(infer_px_directory)
        if "PX_output_dir" not in df.columns:
            df["PX_output_dir"] = df["PX_dir"].map(lambda x: str(ROOT / str(x)))
        rows.append(df)
    if not rows:
        return pd.DataFrame()
    out = pd.concat(rows, ignore_index=True)
    out = out.sort_values(["electrophile_type", "k0_source_workbook"]).drop_duplicates("electrophile_type", keep="first")
    return out.reset_index(drop=True)


def build_water_nu1_rows(k0_summary):
    if k0_summary.empty:
        return pd.DataFrame()
    rows = []
    for _, row in k0_summary.iterrows():
        px = row.get("electrophile_type", np.nan)
        px_dir = row.get("PX_dir", infer_px_directory(px))
        rows.append({
            "PX_directory": px_dir,
            "dataset": "water_hydroxide",
            "PX": px,
            "nu_id": "Nu1",
            "nu_name": "water",
            "rate_constant_source": "k0_water_hydroxide",
            "PX_pKa": row.get("electrophile_pKa", np.nan),
            "A_source_workbook": row.get("k0_source_workbook", np.nan),
            "A_fit_success": row.get("fit_success", False),
            "A_failure_reason": row.get("failure_reason", np.nan),
            "A_plot_path": row.get("plot_path", np.nan),
            "A_n_successful_k2_app": row.get("n_fit_inliers", row.get("n_k0_total_usable", np.nan)),
            "A_n_low_region_points": np.nan,
            "A_n_transition_region_points": np.nan,
            "A_n_high_region_points": row.get("high_pH_k0_n", np.nan),
            "A_k2_low_constrained_by_data": np.nan,
            "A_fit_k2_low": row.get("k0_low", np.nan),
            "A_fit_k2_low_lo95": row.get("k0_low_lo95", np.nan),
            "A_fit_k2_low_hi95": row.get("k0_low_hi95", np.nan),
            "A_fit_k2_high": row.get("k0_high", np.nan),
            "A_fit_k2_high_lo95": row.get("k0_high_lo95", np.nan),
            "A_fit_k2_high_hi95": row.get("k0_high_hi95", np.nan),
            "A_high_pH_k2_app_mean": row.get("high_pH_k0_mean", np.nan),
            "A_high_pH_k2_app_median": row.get("high_pH_k0_median", np.nan),
            "A_high_pH_k2_app_sd": row.get("high_pH_k0_sd", np.nan),
            "A_high_pH_k2_app_n": row.get("high_pH_k0_n", np.nan),
            "A_fit_r2_log": row.get("r2_log", np.nan),
            "A_fit_rmse_log": row.get("rmse_log", np.nan),
            "A_first_stage_total_pH_groups": row.get("n_fit_total", np.nan),
            "A_first_stage_successful_pH_groups": row.get("n_fit_inliers", np.nan),
        })
    return pd.DataFrame(rows)


def mask_failed_fit_values(df, success_col, value_cols):
    out = df.copy()
    if success_col not in out.columns:
        return out
    failed = ~truthy(out[success_col])
    cols = [c for c in value_cols if c in out.columns]
    out.loc[failed, cols] = np.nan
    return out


def coalesce_columns(df, primary, secondary):
    if primary in df.columns and secondary in df.columns:
        df[primary] = df[primary].combine_first(df[secondary])
        df = df.drop(columns=[secondary])
    elif secondary in df.columns and primary not in df.columns:
        df = df.rename(columns={secondary: primary})
    return df


## Read all result workbooks

The notebook reads the final-summary sheets from each analysis workbook:

- `strategy_A_ph_rate`
- `strategy_B_global`
- `strategy_A_kobs_to_k2_app`

The first-stage `strategy_A_kobs_to_k2_app` sheet is used only for simple counting and
diagnostics in the final summary.

In [10]:
result_paths = discover_result_workbooks()
print(f"Found {len(result_paths)} result workbook(s).")
for p in result_paths:
    print("-", p)

a_ph_frames = []
b_global_frames = []
a_kobs_frames = []
source_rows = []

for path in result_paths:
    meta = parse_result_path(path)
    source_rows.append(meta)
    a_ph_frames.append(with_common_metadata(read_sheet(path, "strategy_A_ph_rate"), path))
    b_global_frames.append(with_common_metadata(read_sheet(path, "strategy_B_global"), path))
    a_kobs_frames.append(with_common_metadata(read_sheet(path, "strategy_A_kobs_to_k2_app"), path))

a_ph = pd.concat([x for x in a_ph_frames if not x.empty], ignore_index=True) if any(not x.empty for x in a_ph_frames) else pd.DataFrame()
b_global = pd.concat([x for x in b_global_frames if not x.empty], ignore_index=True) if any(not x.empty for x in b_global_frames) else pd.DataFrame()
a_kobs = pd.concat([x for x in a_kobs_frames if not x.empty], ignore_index=True) if any(not x.empty for x in a_kobs_frames) else pd.DataFrame()
source_workbooks = pd.DataFrame(source_rows)
k0_summary = read_k0_summary_workbooks()

print(f"Strategy A pH-rate rows: {len(a_ph)}")
print(f"Strategy B global rows: {len(b_global)}")
print(f"Strategy A kobs -> k2,app rows: {len(a_kobs)}")
print(f"Water Nu1 k0 rows: {len(k0_summary)}")


Found 30 result workbook(s).
- /Users/robertmayer/Desktop/mayerlab_local/phosphorylation_publ/section5_6_kinetics/P5/S1_NS078/k2_next_step_analysis/S1_NS078_k2_next_step_analysis.xlsx
- /Users/robertmayer/Desktop/mayerlab_local/phosphorylation_publ/section5_6_kinetics/P5/S2_NS073/k2_next_step_analysis/S2_NS073_k2_next_step_analysis.xlsx
- /Users/robertmayer/Desktop/mayerlab_local/phosphorylation_publ/section5_6_kinetics/P5/S3_NS063/k2_next_step_analysis/S3_NS063_k2_next_step_analysis.xlsx
- /Users/robertmayer/Desktop/mayerlab_local/phosphorylation_publ/section5_6_kinetics/P5/S4_NS064/k2_next_step_analysis/S4_NS064_k2_next_step_analysis.xlsx
- /Users/robertmayer/Desktop/mayerlab_local/phosphorylation_publ/section5_6_kinetics/P5/S5_NS070/k2_next_step_analysis/S5_NS070_k2_next_step_analysis.xlsx
- /Users/robertmayer/Desktop/mayerlab_local/phosphorylation_publ/section5_6_kinetics/P5/S6_NS088/k2_next_step_analysis/S6_NS088_k2_next_step_analysis.xlsx
- /Users/robertmayer/Desktop/mayerlab_loc

## Build wide summary table

The wide table keeps one row per `PX_directory / dataset / PX / nu_id / nu_name`.
Fitted Strategy A and Strategy B values are blank when `fit_success` is false.
Strategy A high-pH mean/median values are descriptive summaries and are retained
whenever the input workbook contains them.

In [11]:
def build_strategy_a_summary(a_ph):
    if a_ph.empty:
        return pd.DataFrame(columns=KEY_COLS)
    cols = [
        *KEY_COLS, "source_workbook", "electrophile_pKa",
        "fit_success", "failure_reason", "n_successful_k2_app",
        "n_low_region_points", "n_transition_region_points", "n_high_region_points",
        "k2_low_constrained_by_data",
        "k2_low", "k2_low_lo95", "k2_low_hi95",
        "k2_high", "k2_high_lo95", "k2_high_hi95",
        "high_pH_k2_app_mean", "high_pH_k2_app_median", "high_pH_k2_app_sd", "high_pH_k2_app_n",
        "r2_log", "rmse_log", "plot_path",
    ]
    out = a_ph[[c for c in cols if c in a_ph.columns]].copy()
    out = out.rename(columns={
        "source_workbook": "A_source_workbook",
        "electrophile_pKa": "PX_pKa",
        "fit_success": "A_fit_success",
        "failure_reason": "A_failure_reason",
        "n_successful_k2_app": "A_n_successful_k2_app",
        "n_low_region_points": "A_n_low_region_points",
        "n_transition_region_points": "A_n_transition_region_points",
        "n_high_region_points": "A_n_high_region_points",
        "k2_low_constrained_by_data": "A_k2_low_constrained_by_data",
        "k2_low": "A_fit_k2_low",
        "k2_low_lo95": "A_fit_k2_low_lo95",
        "k2_low_hi95": "A_fit_k2_low_hi95",
        "k2_high": "A_fit_k2_high",
        "k2_high_lo95": "A_fit_k2_high_lo95",
        "k2_high_hi95": "A_fit_k2_high_hi95",
        "high_pH_k2_app_mean": "A_high_pH_k2_app_mean",
        "high_pH_k2_app_median": "A_high_pH_k2_app_median",
        "high_pH_k2_app_sd": "A_high_pH_k2_app_sd",
        "high_pH_k2_app_n": "A_high_pH_k2_app_n",
        "r2_log": "A_fit_r2_log",
        "rmse_log": "A_fit_rmse_log",
        "plot_path": "A_plot_path",
    })
    fit_value_cols = [
        "A_fit_k2_low", "A_fit_k2_low_lo95", "A_fit_k2_low_hi95",
        "A_fit_k2_high", "A_fit_k2_high_lo95", "A_fit_k2_high_hi95",
    ]
    out = mask_failed_fit_values(out, "A_fit_success", fit_value_cols)
    return out


def build_strategy_b_summary(b_global):
    if b_global.empty:
        return pd.DataFrame(columns=KEY_COLS)
    cols = [
        *KEY_COLS, "source_workbook", "electrophile_pKa",
        "fit_success", "failure_reason", "n_selected_k_obs", "n_pH_groups",
        "n_low_region_rows", "n_transition_region_rows", "n_high_region_rows",
        "k2_low", "k2_low_lo95", "k2_low_hi95",
        "k2_high", "k2_high_lo95", "k2_high_hi95",
        "r2_log", "rmse_log", "n_fit_inliers", "n_fit_outliers",
        "plot_path", "ph_rate_profile_plot_path",
    ]
    out = b_global[[c for c in cols if c in b_global.columns]].copy()
    out = out.rename(columns={
        "source_workbook": "B_source_workbook",
        "electrophile_pKa": "PX_pKa",
        "fit_success": "B_fit_success",
        "failure_reason": "B_failure_reason",
        "n_selected_k_obs": "B_n_selected_k_obs",
        "n_pH_groups": "B_n_pH_groups",
        "n_low_region_rows": "B_n_low_region_rows",
        "n_transition_region_rows": "B_n_transition_region_rows",
        "n_high_region_rows": "B_n_high_region_rows",
        "k2_low": "B_global_k2_low",
        "k2_low_lo95": "B_global_k2_low_lo95",
        "k2_low_hi95": "B_global_k2_low_hi95",
        "k2_high": "B_global_k2_high",
        "k2_high_lo95": "B_global_k2_high_lo95",
        "k2_high_hi95": "B_global_k2_high_hi95",
        "r2_log": "B_fit_r2_log",
        "rmse_log": "B_fit_rmse_log",
        "n_fit_inliers": "B_n_fit_inliers",
        "n_fit_outliers": "B_n_fit_outliers",
        "plot_path": "B_obs_pred_plot_path",
        "ph_rate_profile_plot_path": "B_ph_rate_profile_plot_path",
    })
    fit_value_cols = [
        "B_global_k2_low", "B_global_k2_low_lo95", "B_global_k2_low_hi95",
        "B_global_k2_high", "B_global_k2_high_lo95", "B_global_k2_high_hi95",
    ]
    out = mask_failed_fit_values(out, "B_fit_success", fit_value_cols)
    return out


def build_first_stage_counts(a_kobs):
    if a_kobs.empty:
        return pd.DataFrame(columns=KEY_COLS)
    out = a_kobs.copy()
    out["analysis_success"] = truthy(out.get("analysis_success", pd.Series(False, index=out.index)))
    agg = (
        out.groupby(KEY_COLS, dropna=False)
        .agg(
            A_first_stage_total_pH_groups=("ha_initial_pH", "count"),
            A_first_stage_successful_pH_groups=("analysis_success", "sum"),
            A_first_stage_median_r2=("r2", "median"),
            A_first_stage_median_k2_app_rel_ci_width=("k2_app_rel_ci_width", "median") if "k2_app_rel_ci_width" in out.columns else ("r2", lambda s: np.nan),
        )
        .reset_index()
    )
    return agg


a_summary = build_strategy_a_summary(a_ph)
b_summary = build_strategy_b_summary(b_global)
first_stage_counts = build_first_stage_counts(a_kobs)

summary = pd.merge(a_summary, b_summary, on=KEY_COLS, how="outer", suffixes=("", "_B"))
summary = coalesce_columns(summary, "PX_pKa", "PX_pKa_B")
summary = pd.merge(summary, first_stage_counts, on=KEY_COLS, how="left")
summary["rate_constant_source"] = "k2_nucleophile"

water_nu1_rows = build_water_nu1_rows(k0_summary)
if len(water_nu1_rows):
    summary = pd.concat([summary, water_nu1_rows], ignore_index=True, sort=False)

summary["_nu_sort_key"] = summary["nu_id"].map(nu_sort_key)
summary = (
    summary
    .sort_values(["PX_directory", "PX", "_nu_sort_key", "dataset", "nu_id", "nu_name"], kind="stable")
    .drop(columns=["_nu_sort_key"])
    .reset_index(drop=True)
)

print(f"Summary rows: {len(summary)}")
print(f"Water Nu1 rows added: {len(water_nu1_rows)}")
summary.head()


Summary rows: 137
Water Nu1 rows added: 5


,PX_directory,dataset,PX,nu_id,nu_name,A_source_workbook,PX_pKa,A_fit_success,A_failure_reason,A_n_successful_k2_app,...,B_fit_rmse_log,B_n_fit_inliers,B_n_fit_outliers,B_obs_pred_plot_path,B_ph_rate_profile_plot_path,A_first_stage_total_pH_groups,A_first_stage_successful_pH_groups,A_first_stage_median_r2,A_first_stage_median_k2_app_rel_ci_width,rate_constant_source
0,P5,water_hydroxide,P5,Nu1,water,/Users/robertmayer/Desktop/mayerlab_local/phos...,4.5,True,NaN,345.0,...,NaN,NaN,NaN,NaN,NaN,361,345,NaN,NaN,k0_water_hydroxide
1,P5,S1_NS078,P5,Nu2,pyridine,/Users/robertmayer/Desktop/mayerlab_local/phos...,4.5,True,NaN,9.0,...,0.011831,89.0,5.0,/Users/robertmayer/Desktop/mayerlab_local/phos...,/Users/robertmayer/Desktop/mayerlab_local/phos...,16,9,0.745235,0.880788,k2_nucleophile
2,P5,S1_NS078,P5,Nu3,3-methylpyridine,/Users/robertmayer/Desktop/mayerlab_local/phos...,4.5,True,NaN,11.0,...,0.017447,89.0,6.0,/Users/robertmayer/Desktop/mayerlab_local/phos...,/Users/robertmayer/Desktop/mayerlab_local/phos...,16,11,0.938432,0.482643,k2_nucleophile
3,P5,S1_NS078,P5,Nu4,4-methylpyridine,/Users/robertmayer/Desktop/mayerlab_local/phos...,4.5,True,NaN,8.0,...,0.019641,86.0,9.0,/Users/robertmayer/Desktop/mayerlab_local/phos...,/Users/robertmayer/Desktop/mayerlab_local/phos...,16,8,0.738733,1.026355,k2_nucleophile
4,P5,S1_NS078,P5,Nu5,"3,4-lutidine",/Users/robertmayer/Desktop/mayerlab_local/phos...,4.5,True,NaN,11.0,...,0.008773,88.0,7.0,/Users/robertmayer/Desktop/mayerlab_local/phos...,/Users/robertmayer/Desktop/mayerlab_local/phos...,16,11,0.870488,0.584708,k2_nucleophile


## Build concise SI-style summary

The Excel output is intentionally compact. The `summary` sheet keeps only the columns
needed for reporting final values: identifiers, Strategy A fitted values, Strategy A
high-pH mean/median values, and Strategy B global-fit values. The only retained count
columns are the point counts used for the Strategy A and Strategy B fits.

The final $k_{2,\mathrm{high}}$ value, including the `Nu1` water entry, is selected by comparing eligible Strategy A and Strategy B
fit estimates by relative CI width. If neither fit-based value is eligible, the
Strategy A high-pH median can be used as a fallback when enough high-pH points are
available and the scatter is acceptable.

The `metadata` sheet keeps source workbook paths, plot paths, failure reasons, and
selected diagnostics so the concise table can still be audited.

In [12]:
summary_cols = [
    "PX_directory", "dataset", "PX", "nu_id", "nu_name", "rate_constant_source", "PX_pKa",
    "A_fit_success", "A_n_successful_k2_app",
    "A_fit_k2_low", "A_fit_k2_low_lo95", "A_fit_k2_low_hi95",
    "A_fit_k2_high", "A_fit_k2_high_lo95", "A_fit_k2_high_hi95",
    "A_high_pH_k2_app_mean", "A_high_pH_k2_app_median", "A_high_pH_k2_app_sd",
    "B_fit_success", "B_n_fit_inliers",
    "B_global_k2_low", "B_global_k2_low_lo95", "B_global_k2_low_hi95",
    "B_global_k2_high", "B_global_k2_high_lo95", "B_global_k2_high_hi95",
]
summary_si = summary[[c for c in summary_cols if c in summary.columns]].copy()
summary_si = summary_si.rename(columns={
    "A_n_successful_k2_app": "A_n_k2_points_fit",
    "B_n_fit_inliers": "B_n_kobs_points_fit",
})
for success_col in ["A_fit_success", "B_fit_success"]:
    if success_col in summary_si.columns:
        summary_si[success_col] = truthy(summary_si[success_col])

FINAL_FIT_MAX_REL_ERROR = 3.0
FINAL_MEDIAN_MIN_N = 3
FINAL_MEDIAN_MAX_REL_SD = 3.0


def relative_ci_width(value, lo, hi):
    if pd.isna(value) or pd.isna(lo) or pd.isna(hi) or value <= 0:
        return np.nan
    return (hi - lo) / abs(value)


def positive_number(value):
    return pd.notna(value) and np.isfinite(value) and value > 0


def choose_final_k2_high(row):
    candidates = []

    if row.get("rate_constant_source") == "k0_water_hydroxide":
        a_rel = relative_ci_width(
            row.get("A_fit_k2_high", np.nan),
            row.get("A_fit_k2_high_lo95", np.nan),
            row.get("A_fit_k2_high_hi95", np.nan),
        )
        if (
            bool(row.get("A_fit_success", False))
            and positive_number(row.get("A_fit_k2_high", np.nan))
        ):
            return pd.Series({
                "final_k2_high": row.get("A_fit_k2_high", np.nan),
                "final_k2_high_lo95": row.get("A_fit_k2_high_lo95", np.nan),
                "final_k2_high_hi95": row.get("A_fit_k2_high_hi95", np.nan),
                "final_k2_high_method": "k0_water_hydroxide_fit",
                "final_k2_high_rel_error": a_rel,
                "A_fit_k2_high_rel_error": a_rel,
                "B_global_k2_high_rel_error": np.nan,
                "A_high_pH_median_rel_sd": np.nan,
            })
        return pd.Series({
            "final_k2_high": np.nan,
            "final_k2_high_lo95": np.nan,
            "final_k2_high_hi95": np.nan,
            "final_k2_high_method": "",
            "final_k2_high_rel_error": np.nan,
            "A_fit_k2_high_rel_error": a_rel,
            "B_global_k2_high_rel_error": np.nan,
            "A_high_pH_median_rel_sd": np.nan,
        })

    a_rel = relative_ci_width(
        row.get("A_fit_k2_high", np.nan),
        row.get("A_fit_k2_high_lo95", np.nan),
        row.get("A_fit_k2_high_hi95", np.nan),
    )
    if (
        bool(row.get("A_fit_success", False))
        and positive_number(row.get("A_fit_k2_high", np.nan))
        and pd.notna(a_rel)
        and a_rel <= FINAL_FIT_MAX_REL_ERROR
    ):
        candidates.append({
            "method": "A_fit",
            "value": row.get("A_fit_k2_high", np.nan),
            "lo95": row.get("A_fit_k2_high_lo95", np.nan),
            "hi95": row.get("A_fit_k2_high_hi95", np.nan),
            "rel_error": a_rel,
            "rank_group": 1,
        })

    b_rel = relative_ci_width(
        row.get("B_global_k2_high", np.nan),
        row.get("B_global_k2_high_lo95", np.nan),
        row.get("B_global_k2_high_hi95", np.nan),
    )
    if (
        bool(row.get("B_fit_success", False))
        and positive_number(row.get("B_global_k2_high", np.nan))
        and pd.notna(b_rel)
        and b_rel <= FINAL_FIT_MAX_REL_ERROR
    ):
        candidates.append({
            "method": "B_global",
            "value": row.get("B_global_k2_high", np.nan),
            "lo95": row.get("B_global_k2_high_lo95", np.nan),
            "hi95": row.get("B_global_k2_high_hi95", np.nan),
            "rel_error": b_rel,
            "rank_group": 1,
        })

    fit_candidates = [c for c in candidates if c["rank_group"] == 1]
    if fit_candidates:
        best = sorted(fit_candidates, key=lambda c: c["rel_error"])[0]
        return pd.Series({
            "final_k2_high": best["value"],
            "final_k2_high_lo95": best["lo95"],
            "final_k2_high_hi95": best["hi95"],
            "final_k2_high_method": best["method"],
            "final_k2_high_rel_error": best["rel_error"],
            "A_fit_k2_high_rel_error": a_rel,
            "B_global_k2_high_rel_error": b_rel,
            "A_high_pH_median_rel_sd": np.nan,
        })

    median = row.get("A_high_pH_k2_app_median", np.nan)
    sd = row.get("A_high_pH_k2_app_sd", np.nan)
    n = row.get("A_high_pH_k2_app_n", np.nan)
    median_rel_sd = sd / abs(median) if positive_number(median) and pd.notna(sd) else np.nan
    if (
        positive_number(median)
        and pd.notna(n)
        and n >= FINAL_MEDIAN_MIN_N
        and pd.notna(median_rel_sd)
        and median_rel_sd <= FINAL_MEDIAN_MAX_REL_SD
    ):
        return pd.Series({
            "final_k2_high": median,
            "final_k2_high_lo95": np.nan,
            "final_k2_high_hi95": np.nan,
            "final_k2_high_method": "A_high_pH_median",
            "final_k2_high_rel_error": median_rel_sd,
            "A_fit_k2_high_rel_error": a_rel,
            "B_global_k2_high_rel_error": b_rel,
            "A_high_pH_median_rel_sd": median_rel_sd,
        })

    return pd.Series({
        "final_k2_high": np.nan,
        "final_k2_high_lo95": np.nan,
        "final_k2_high_hi95": np.nan,
        "final_k2_high_method": "",
        "final_k2_high_rel_error": np.nan,
        "A_fit_k2_high_rel_error": a_rel,
        "B_global_k2_high_rel_error": b_rel,
        "A_high_pH_median_rel_sd": median_rel_sd,
    })


final_selection = summary.apply(choose_final_k2_high, axis=1)
summary = pd.concat([summary, final_selection], axis=1)
summary_si = pd.concat([summary_si, final_selection[[
    "final_k2_high", "final_k2_high_lo95", "final_k2_high_hi95",
    "final_k2_high_method", "final_k2_high_rel_error",
]]], axis=1)

front_cols = [
    "PX_directory", "dataset", "PX", "nu_id", "nu_name", "rate_constant_source", "PX_pKa",
    "final_k2_high", "final_k2_high_lo95", "final_k2_high_hi95",
    "final_k2_high_method", "final_k2_high_rel_error",
]
summary_si = summary_si[[c for c in front_cols if c in summary_si.columns] + [c for c in summary_si.columns if c not in front_cols]]

metadata_cols = [
    "PX_directory", "dataset", "PX", "nu_id", "nu_name", "rate_constant_source", "PX_pKa",
    "final_k2_high", "final_k2_high_lo95", "final_k2_high_hi95",
    "final_k2_high_method", "final_k2_high_rel_error",
    "A_fit_k2_high_rel_error", "B_global_k2_high_rel_error", "A_high_pH_median_rel_sd",
    "A_source_workbook", "A_fit_success", "A_failure_reason", "A_plot_path",
    "A_n_successful_k2_app", "A_n_low_region_points", "A_n_transition_region_points",
    "A_n_high_region_points", "A_k2_low_constrained_by_data",
    "A_fit_r2_log", "A_fit_rmse_log",
    "B_source_workbook", "B_fit_success", "B_failure_reason",
    "B_obs_pred_plot_path", "B_ph_rate_profile_plot_path",
    "B_n_selected_k_obs", "B_n_fit_inliers", "B_n_fit_outliers",
    "B_n_pH_groups", "B_fit_r2_log", "B_fit_rmse_log",
    "A_first_stage_total_pH_groups", "A_first_stage_successful_pH_groups",
    "A_first_stage_median_r2", "A_first_stage_median_k2_app_rel_ci_width",
]
metadata = summary[[c for c in metadata_cols if c in summary.columns]].copy()

print(f"Concise summary rows: {len(summary_si)}")
summary_si.head()

Concise summary rows: 137


/var/folders/1g/tqprlkrd1bx9j8npw4wpqsh00000gn/T/ipykernel_11126/197948419.py:40: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return series.fillna(False).astype(bool)
/var/folders/1g/tqprlkrd1bx9j8npw4wpqsh00000gn/T/ipykernel_11126/197948419.py:40: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return series.fillna(False).astype(bool)


,PX_directory,dataset,PX,nu_id,nu_name,rate_constant_source,PX_pKa,final_k2_high,final_k2_high_lo95,final_k2_high_hi95,...,A_high_pH_k2_app_median,A_high_pH_k2_app_sd,B_fit_success,B_n_kobs_points_fit,B_global_k2_low,B_global_k2_low_lo95,B_global_k2_low_hi95,B_global_k2_high,B_global_k2_high_lo95,B_global_k2_high_hi95
0,P5,water_hydroxide,P5,Nu1,water,k0_water_hydroxide,4.5,0.000024,0.000024,0.000024,...,0.000024,0.000004,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,P5,S1_NS078,P5,Nu2,pyridine,k2_nucleophile,4.5,0.000113,0.000095,0.000130,...,0.000099,0.000090,True,89.0,2.386243e-18,2.658249e-28,1.207259e-04,0.000113,0.000095,0.000130
2,P5,S1_NS078,P5,Nu3,3-methylpyridine,k2_nucleophile,4.5,0.000106,0.000084,0.000126,...,0.000117,0.000081,True,89.0,1.455637e-22,2.326674e-29,1.419101e-16,0.000106,0.000084,0.000126
3,P5,S1_NS078,P5,Nu4,4-methylpyridine,k2_nucleophile,4.5,0.000134,0.000120,0.000146,...,0.000139,0.000034,True,86.0,5.031154e-04,2.008541e-21,5.181029e-03,0.000110,0.000081,0.000133
4,P5,S1_NS078,P5,Nu5,"3,4-lutidine",k2_nucleophile,4.5,0.000118,0.000107,0.000130,...,0.000118,0.000033,True,88.0,2.056205e-15,5.312179e-23,1.474106e-03,0.000125,0.000112,0.000140


## Write summary workbook

Only an Excel workbook is written. It contains:

- `summary`: concise SI-style final values
- `metadata`: source files, plot paths, failure reasons, and diagnostics

In [13]:
def autosize_worksheet(ws, max_width=45):
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
    for column_cells in ws.columns:
        max_len = max(len(str(cell.value)) if cell.value is not None else 0 for cell in column_cells)
        width = min(max(max_len + 2, 10), max_width)
        ws.column_dimensions[column_cells[0].column_letter].width = width


def format_summary_sheet(ws):
    for cell in ws[1]:
        if cell.value and ("k2" in str(cell.value).lower()):
            col_letter = cell.column_letter
            for data_cell in ws[col_letter][1:]:
                data_cell.number_format = "0.00E+00"
        if cell.value and str(cell.value).endswith("_points_fit"):
            col_letter = cell.column_letter
            for data_cell in ws[col_letter][1:]:
                data_cell.number_format = "0"
        if cell.value and str(cell.value).endswith("_success"):
            col_letter = cell.column_letter
            for data_cell in ws[col_letter][1:]:
                data_cell.number_format = "General"


output_xlsx = OUTPUT_DIR / "stage6_k2_high_selection_summary.xlsx"

with pd.ExcelWriter(output_xlsx, engine="openpyxl") as writer:
    summary_si.to_excel(writer, sheet_name="summary", index=False)
    metadata.to_excel(writer, sheet_name="metadata", index=False)
    for ws in writer.book.worksheets:
        autosize_worksheet(ws)
    format_summary_sheet(writer.book["summary"])

print(f"Wrote {output_xlsx}")

Wrote /Users/robertmayer/Desktop/mayerlab_local/phosphorylation_publ/section5_6_kinetics/stage6_k2_high_selection_summary.xlsx


## Quick overview

This compact count table helps catch missing workbooks or unexpectedly failed fits.

In [14]:
overview = (
    summary
    .assign(
        A_fit_success=truthy(summary.get("A_fit_success", pd.Series(False, index=summary.index))),
        B_fit_success=truthy(summary.get("B_fit_success", pd.Series(False, index=summary.index))),
    )
    .groupby(["PX_directory", "PX"], dropna=False)
    .agg(
        n_nucleophile_dataset_rows=("nu_id", "count"),
        n_strategy_A_fit_success=("A_fit_success", "sum"),
        n_strategy_B_fit_success=("B_fit_success", "sum"),
    )
    .reset_index()
)
overview

/var/folders/1g/tqprlkrd1bx9j8npw4wpqsh00000gn/T/ipykernel_11126/197948419.py:40: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return series.fillna(False).astype(bool)
/var/folders/1g/tqprlkrd1bx9j8npw4wpqsh00000gn/T/ipykernel_11126/197948419.py:40: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return series.fillna(False).astype(bool)


,PX_directory,PX,n_nucleophile_dataset_rows,n_strategy_A_fit_success,n_strategy_B_fit_success
0,P5,P5,29,23,28
1,P6,P6,26,15,25
2,P7,P7,28,20,27
3,P8,P8,29,21,28
4,P9,P9,25,14,24
